# Bài 3: Đọc biển số xe (notebook mẫu)

Mỗi ảnh chứa một biển số xe Việt Nam: biển ô tô dài một dòng, biển ô tô vuông hai dòng hoặc biển xe máy hai dòng. Cần đọc
các chữ cái và chữ số trên biển, đọc dòng trên trước rồi tới dòng dưới, bỏ dấu gạch và dấu chấm. Ví dụ biển xe máy có dòng
trên `59-X1` và dòng dưới `123.45` được đọc là `59X112345`. Điểm là tỉ lệ ảnh được đọc đúng hoàn toàn.

Baseline trong notebook này làm ba bước:

1. Đưa mọi ảnh về cùng kích thước, cao 96 điểm ảnh và rộng 192 điểm ảnh.
2. Lấy model ResNet-18 của torchvision đã train sẵn trên bộ ảnh ImageNet. Lớp cuối của model được thay bằng 9 bộ phân loại,
   mỗi bộ đoán ký tự ở một vị trí của chuỗi. Biển dài nhất có 9 ký tự. Với biển ngắn hơn, các vị trí thừa có nhãn "trống".
3. Train toàn bộ model 12 lượt qua dữ liệu (12 epoch) trên 90 % ảnh train, giữ 10 % còn lại làm validation, rồi đọc ảnh
   của tập test.

Notebook cần GPU, và chạy trong khoảng 15 đến 20 phút trên một GPU loại T4. Không có GPU thì notebook vẫn chạy được trên CPU
nhưng chậm hơn nhiều. Ban tổ chức đo được tỉ lệ đọc đúng khoảng **0,75** trên validation và khoảng **0,75** trên tập test.
Một số phép tính trên GPU không cho kết quả giống hệt nhau giữa các lần chạy, nên chạy lại notebook mẫu có thể cho điểm chênh
nhau tới khoảng 0,03. Vì vậy, một thay đổi làm điểm validation tăng ít hơn mức này chưa chắc đã có tác dụng thật.

**Quy định khi làm bài, theo điều kiện của vòng thi OLP AI**

- Chỉ dùng Python và các thư viện trong danh sách của thể lệ. Không cài thêm thư viện bằng `pip` hay `conda`.
- Được dùng trọng số pretrained có sẵn trong torchvision, ví dụ ResNet hay ConvNeXt đã train trên ImageNet. Không dùng
  model đọc chữ (OCR) có sẵn, model ngôn ngữ và thị giác lớn, hay LLM.
- Chỉ dùng ảnh của đề. Được biến đổi ảnh train để có thêm dữ liệu, kể cả cắt ký tự từ ảnh train rồi dán ghép lại. Không
  được tự vẽ biển số mới bằng font chữ, và không train trên ảnh của tập test.
- Tệp kết quả phải được tạo ra bằng cách chạy notebook tuần tự từ ô đầu tiên đến ô cuối cùng. Mỗi khi tạo tệp để nộp, hãy
  chọn **Restart & Run All** (trên Kaggle là **Save Version**), để số thứ tự chạy của các ô code là 1, 2, 3, … từ trên xuống
  dưới.
- Không sửa tay dữ liệu đầu vào hay tệp kết quả.

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

# DATA_DIR là thư mục chứa dữ liệu của đề.
# Khi làm trên Kaggle, hãy sửa dòng dưới thành DATA_DIR = Path("/kaggle/input/<tên dataset>/public").
DATA_DIR = Path(".")

# Nếu dữ liệu không nằm cạnh notebook mà nằm trong thư mục public, tức thư mục tạo ra khi giải nén input.zip,
# thì đọc dữ liệu trong thư mục public.
if not (DATA_DIR / "train.csv").exists() and Path("public/train.csv").exists():
    DATA_DIR = Path("public")

# Tệp scorer.py nằm cùng thư mục với dữ liệu. Hàm score trong tệp này tính điểm giống hệt metric của SkillPixel.
sys.path.insert(0, str(DATA_DIR))
from scorer import score

# Mỗi dòng của train.csv gồm mã ảnh, biển số và số dòng chữ trên biển (cột lines).
train = pd.read_csv(DATA_DIR / "train.csv", dtype=str)
# Tập test chỉ có cột id, là mã của các ảnh cần đọc.
test = pd.read_csv(DATA_DIR / "test.csv", dtype=str)

print(len(train), "ảnh train,", len(test), "ảnh test")
print("số ảnh train theo số dòng chữ trên biển:", train.lines.value_counts().to_dict())
train.head()

## 1. Đọc ảnh và mã hoá nhãn

Mỗi ký tự được đổi thành số thứ tự của nó trong chuỗi `KY_TU`, và số 31 nghĩa là "trống". Ví dụ nhãn `51F12345` có 8 ký tự,
nên vị trí thứ 9 có nhãn "trống".

In [ ]:
# Mọi ảnh được đưa về cùng kích thước, cao 96 điểm ảnh và rộng 192 điểm ảnh.
CAO = 96
RONG = 192

# Biển số trong dữ liệu chỉ dùng 31 ký tự dưới đây. Các chữ I, J, O, Q và W không xuất hiện.
KY_TU = "0123456789ABCDEFGHKLMNPRSTUVXYZ"
# Số 31 là nhãn "trống", dùng cho các vị trí không có ký tự.
TRONG = len(KY_TU)
# Biển dài nhất có 9 ký tự, nên model đoán 9 vị trí.
DO_DAI = 9


def doc_anh(thu_muc, cac_ma_anh):
    """Đọc các ảnh có mã trong cac_ma_anh, trả về một mảng numpy kiểu uint8 dạng (số ảnh, CAO, RONG, 3)."""
    anh = np.zeros((len(cac_ma_anh), CAO, RONG, 3), np.uint8)
    for i, ma_anh in enumerate(cac_ma_anh):
        mot_anh = cv2.imread(str(thu_muc / f"{ma_anh}.jpg"))
        # OpenCV đọc ảnh với các kênh màu theo thứ tự BGR. ResNet được train trên ảnh RGB nên cần đổi thứ tự kênh.
        mot_anh = cv2.cvtColor(mot_anh, cv2.COLOR_BGR2RGB)
        # Hàm cv2.resize nhận kích thước theo thứ tự chiều rộng rồi chiều cao.
        anh[i] = cv2.resize(mot_anh, (RONG, CAO), interpolation=cv2.INTER_AREA)
    return anh


def ma_hoa(bien_so):
    """Đổi một biển số thành 9 số nguyên. Mỗi ký tự thành số thứ tự của nó trong KY_TU, các vị trí thừa thành TRONG."""
    cac_so = []
    for ky_tu in bien_so:
        cac_so.append(KY_TU.index(ky_tu))
    while len(cac_so) < DO_DAI:
        cac_so.append(TRONG)
    return cac_so


def giai_ma(hang):
    """Đổi 9 số mà model dự đoán thành chuỗi biển số, bỏ qua các vị trí có nhãn trống."""
    bien_so = ""
    for so in hang:
        if so != TRONG:
            bien_so += KY_TU[so]
    return bien_so


bat_dau = time.time()
X = doc_anh(DATA_DIR / "train", train.id)
X_test = doc_anh(DATA_DIR / "test", test.id)
Y = np.array([ma_hoa(bien_so) for bien_so in train.plate])
print("đọc xong", len(X) + len(X_test), "ảnh sau", round(time.time() - bat_dau), "giây")
print(train.plate[0], "được mã hoá thành", Y[0])

## 2. Model và cách train

Ảnh được chuẩn hoá theo trung bình và độ lệch chuẩn của ImageNet, vì ResNet-18 đã được train với ảnh chuẩn hoá như vậy.
Hàm mất mát là cross-entropy, tính trung bình trên 9 vị trí. Tốc độ học tăng dần ở 10 % số bước đầu rồi giảm dần về gần 0
(lịch OneCycleLR).

In [ ]:
# Dùng GPU nếu có, nếu không thì dùng CPU.
thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"

# Cố định hạt giống ngẫu nhiên của torch, để trọng số ban đầu của lớp cuối giống nhau giữa các lần chạy.
torch.manual_seed(0)

# Trung bình và độ lệch chuẩn của ba kênh màu trên bộ ảnh ImageNet. Các số được nhân với 255 vì điểm ảnh ở đây có giá trị
# từ 0 đến 255.
TRUNG_BINH = torch.tensor([0.485, 0.456, 0.406], device=thiet_bi).view(1, 3, 1, 1) * 255
DO_LECH = torch.tensor([0.229, 0.224, 0.225], device=thiet_bi).view(1, 3, 1, 1) * 255


def chuan_hoa(anh):
    """Đổi mảng ảnh uint8 dạng (số ảnh, cao, rộng, 3) thành tensor dạng (số ảnh, 3, cao, rộng) đã chuẩn hoá."""
    x = torch.from_numpy(anh).to(thiet_bi)
    # Model của PyTorch cần chiều kênh màu đứng ngay sau chiều số ảnh.
    x = x.permute(0, 3, 1, 2).float()
    return (x - TRUNG_BINH) / DO_LECH


def huan_luyen(X, Y, so_epoch=12, kich_thuoc_lo=128):
    """Train ResNet-18 trên ảnh X với nhãn Y rồi trả về model đã train.

    Mỗi epoch duyệt qua toàn bộ ảnh train một lần. Mỗi bước train dùng một lô gồm kich_thuoc_lo ảnh.
    """
    # Lấy ResNet-18 đã train trên ImageNet. Lớp cuối được thay bằng một lớp mới cho ra 9 nhóm 32 số với mỗi ảnh, mỗi nhóm
    # là điểm số của 32 lớp ở một vị trí. 32 lớp gồm 31 ký tự và nhãn trống.
    model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(512, DO_DAI * (TRONG + 1))
    model = model.to(thiet_bi)

    # AdamW là thuật toán cập nhật trọng số. Lịch OneCycleLR tăng tốc độ học trong 10 % số bước đầu, rồi giảm dần về gần 0
    # ở bước cuối cùng.
    bo_toi_uu = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Số lô của mỗi epoch được làm tròn lên, vì lô cuối có thể có ít ảnh hơn.
    so_lo_moi_epoch = (len(X) + kich_thuoc_lo - 1) // kich_thuoc_lo
    so_buoc = so_epoch * so_lo_moi_epoch
    lich_hoc = torch.optim.lr_scheduler.OneCycleLR(bo_toi_uu, max_lr=1e-3, total_steps=so_buoc, pct_start=0.1)

    # Bộ sinh số ngẫu nhiên của numpy, dùng để xáo thứ tự ảnh ở mỗi epoch.
    rng = np.random.default_rng(0)
    bat_dau = time.time()
    for epoch in range(so_epoch):
        model.train()
        thu_tu = rng.permutation(len(X))
        tong_loss = 0.0
        for dau_lo in range(0, len(X), kich_thuoc_lo):
            # Lấy một lô ảnh và nhãn theo thứ tự đã xáo.
            chi_so_lo = thu_tu[dau_lo:dau_lo + kich_thuoc_lo]
            anh_lo = chuan_hoa(X[chi_so_lo])
            nhan_lo = torch.from_numpy(Y[chi_so_lo]).to(thiet_bi)

            # Đổi đầu ra của model về dạng (số ảnh, 9, 32), để mỗi vị trí có 32 điểm số riêng.
            logits = model(anh_lo).view(-1, DO_DAI, TRONG + 1)

            # Cross-entropy được tính ở từng vị trí của từng ảnh rồi lấy trung bình.
            loss = nn.functional.cross_entropy(logits.reshape(-1, TRONG + 1), nhan_lo.reshape(-1))

            # Tính gradient, cập nhật trọng số, rồi chuyển tốc độ học sang giá trị của bước tiếp theo.
            bo_toi_uu.zero_grad()
            loss.backward()
            bo_toi_uu.step()
            lich_hoc.step()

            tong_loss += loss.item() * len(chi_so_lo)
        print(f"epoch {epoch + 1}: loss {tong_loss / len(X):.4f}, {time.time() - bat_dau:.0f} giây")
    return model


def doc_bien(model, X):
    """Đọc biển số của mọi ảnh trong X và trả về danh sách các chuỗi biển số."""
    model.eval()
    ket_qua = []
    # Khi dự đoán thì không cần gradient. Tắt việc tính gradient giúp chạy nhanh hơn và tốn ít bộ nhớ hơn.
    with torch.no_grad():
        # Đọc 256 ảnh mỗi lần để không bị thiếu bộ nhớ GPU.
        for dau_lo in range(0, len(X), 256):
            logits = model(chuan_hoa(X[dau_lo:dau_lo + 256])).view(-1, DO_DAI, TRONG + 1)
            # Ở mỗi vị trí, chọn lớp có điểm số cao nhất.
            du_doan = logits.argmax(-1).cpu().numpy()
            for hang in du_doan:
                ket_qua.append(giai_ma(hang))
    return ket_qua

## 3. Train và tự đánh giá trên validation

Notebook giữ lại 10 % ảnh train ngẫu nhiên làm validation và train trên phần còn lại.

In [ ]:
# Xáo thứ tự ảnh train với hạt giống cố định. 10 % số ảnh đầu tiên sau khi xáo làm validation, phần còn lại để train.
thu_tu = np.random.default_rng(0).permutation(len(train))
so_anh_validation = len(train) // 10
chi_so_validation = thu_tu[:so_anh_validation]
chi_so_train = thu_tu[so_anh_validation:]

model = huan_luyen(X[chi_so_train], Y[chi_so_train])

# Đáp án validation có hai cột id và plate, giống tệp đáp án mà SkillPixel dùng để chấm.
dap_an_validation = train.iloc[chi_so_validation][["id", "plate"]]
du_doan_validation = pd.DataFrame({
    "id": dap_an_validation.id.values,
    "plate": doc_bien(model, X[chi_so_validation]),
})
diem = score(dap_an_validation, du_doan_validation, "id")
print("Tỉ lệ đọc đúng trên validation:", round(diem, 4))

## 4. Dự đoán tập test

Ô dưới đây dùng chính model vừa train để đọc ảnh của tập test, rồi ghi tệp `bai3_submission.csv`. Đây là tệp để nộp lên
SkillPixel.

In [ ]:
bai_nop = pd.DataFrame({
    "id": test.id,
    "plate": doc_bien(model, X_test),
})
bai_nop.to_csv("bai3_submission.csv", index=False)
print("Đã ghi bai3_submission.csv:", len(bai_nop), "dòng")
bai_nop.head()

## 5. Nộp bài

Tệp `bai3_submission.csv` do ô trên ghi ra là tệp để nộp lên SkillPixel. SkillPixel chấm mỗi lần nộp trên toàn bộ tập test. Mỗi lần nộp,
hãy lưu lại tệp csv vừa nộp cùng notebook đã chạy tạo ra nó.

Trước 6:00, đội chọn một lần nộp làm bài nộp cuối, thường là lần có điểm cao nhất. Tệp nộp bài cuối là `bai3_<tên đội>.zip`. Tệp zip này
chứa đúng hai tệp, là tệp csv của lần nộp được chọn và notebook đã tạo ra tệp csv đó, còn nguyên output. Trước khi nộp qua
địa chỉ mà ban tổ chức công bố, hãy kiểm tra tệp zip bằng lệnh dưới đây. Trong lệnh này, `public` là thư mục tạo ra khi giải
nén `input.zip`.

```
python3 public/kiem_tra_bai_nop.py bai3_<tên đội>.zip --du-lieu public
```